# 3.2 Designing and comparing heuristics: practice notebook

Where good heuristics come from, how to tell which of two heuristics is better, and how to measure one. The 8 puzzle, relaxed problems, dominance, pattern databases and the effective branching factor.

How to use it: run the cells from top to bottom (Shift + Enter). First read and run the examples. Then try each task in its starter cell and run the check cell under it. The full solutions are at the end, so try on your own first.

Everything here is plain Python 3. No extra packages are needed.

## Setup

In [ ]:
# A small helper. check(tests) runs each test of a task and tells you which ones pass.
# Each test is a pair: the call to make, and the answer we expect.
import json

def check(tests):
    passed = 0
    for call, expected in tests:
        try:
            got = eval(call)
        except Exception as error:
            got = "error: " + str(error)
        same = json.dumps(got, sort_keys=True, default=repr) == json.dumps(expected, sort_keys=True, default=repr)
        if same:
            passed = passed + 1
            print("PASS ", call)
        else:
            print("FAIL ", call, " expected", expected, " but got", got)
    if passed == len(tests):
        print("All", len(tests), "tests pass. Well done!")
    else:
        print(passed, "of", len(tests), "tests pass. Keep going, or look at the solution below.")

## Examples from the chapter

### Misplaced tiles and Manhattan distance on the 8 puzzle

In [ ]:
# Two heuristics for the 8 puzzle. A state is a list of 9 numbers read row by row, 0 is the blank.
start = [7, 2, 4,
         5, 0, 6,
         8, 3, 1]
goal = [0, 1, 2,
        3, 4, 5,
        6, 7, 8]

def misplaced(state):                     # h1: tiles not in their goal square
    count = 0
    for i in range(9):
        if state[i] != 0 and state[i] != goal[i]:
            count = count + 1
    return count

def manhattan(state):                     # h2: total up, down, left, right distance of every tile
    total = 0
    for i in range(9):
        tile = state[i]
        if tile != 0:
            j = goal.index(tile)
            total = total + abs(i // 3 - j // 3) + abs(i % 3 - j % 3)
    return total

print("h1 =", misplaced(start))
print("h2 =", manhattan(start))

### Finding the effective branching factor

In [ ]:
# Effective branching factor: find b so that 1 + b + b^2 + ... + b^d = N + 1.
# We use halving (bisection): the left side grows with b, so we can squeeze b between two bounds.
def tree_size(b, d):
    total = 0
    for i in range(d + 1):
        total = total + b ** i
    return total

def effective_b(N, d):
    low = 1.0
    high = 10.0
    for step in range(20):
        mid = (low + high) / 2
        if tree_size(mid, d) < N + 1:
            low = mid
        else:
            high = mid
    return round(low, 2)

print("N = 52, d = 5:", effective_b(52, 5))
# try effective_b(57, 12) and effective_b(164, 12), the A* runs with h2 and h1 at depth 12

## Your turn

### Task 1: Misplaced tiles and Manhattan distance

Write `heuristics(state, goal)` for the 8 puzzle. Both are lists of 9 numbers (0 is the blank). Return [h1, h2]: misplaced tiles and Manhattan distance. Never count the blank.

Hint: Square i is at row i // 3 and column i % 3.

In [ ]:
def heuristics(state, goal):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([('heuristics([7, 2, 4, 5, 0, 6, 8, 3, 1], [0, 1, 2, 3, 4, 5, 6, 7, 8])', [8, 18]), ('heuristics([1, 0, 2, 3, 4, 5, 6, 7, 8], [0, 1, 2, 3, 4, 5, 6, 7, 8])', [1, 1]), ('heuristics([1, 2, 3, 4, 5, 6, 7, 8, 0], [1, 2, 3, 4, 5, 6, 7, 8, 0])', [0, 0])])

### Task 2: Effective branching factor

Write `effective_b(N, d)` that returns b* rounded to 2 decimals, where 1 + b* + ... + (b*)<sup>d</sup> = N + 1. Use bisection between 1 and N + 1.

Hint: The left side grows with b. If it is too small, move the low end up, else move the high end down.

In [ ]:
def effective_b(N, d):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([('effective_b(6, 2)', 2.0), ('effective_b(52, 5)', 1.92), ('effective_b(14, 3)', 2.0), ('effective_b(57, 12)', 1.23)])

## Solutions

Try the tasks yourself first. Running a solution cell replaces your own function with the model one.

### Solution to task 1: Misplaced tiles and Manhattan distance

The first test is the textbook start: 8 and 18.

In [ ]:
def heuristics(state, goal):
    h1 = 0
    h2 = 0
    for i in range(9):
        tile = state[i]
        if tile != 0:
            j = goal.index(tile)
            if i != j:
                h1 = h1 + 1
            h2 = h2 + abs(i // 3 - j // 3) + abs(i % 3 - j % 3)
    return [h1, h2]

In [ ]:
check([('heuristics([7, 2, 4, 5, 0, 6, 8, 3, 1], [0, 1, 2, 3, 4, 5, 6, 7, 8])', [8, 18]), ('heuristics([1, 0, 2, 3, 4, 5, 6, 7, 8], [0, 1, 2, 3, 4, 5, 6, 7, 8])', [1, 1]), ('heuristics([1, 2, 3, 4, 5, 6, 7, 8, 0], [1, 2, 3, 4, 5, 6, 7, 8, 0])', [0, 0])])

### Solution to task 2: Effective branching factor

57 nodes at depth 12 gives b* ≈ 1.23, the value of A*(h2) in our table.

In [ ]:
def effective_b(N, d):
    low = 1.0
    high = N + 1.0
    for step in range(60):
        mid = (low + high) / 2
        total = 0
        for i in range(d + 1):
            total = total + mid ** i
        if total < N + 1:
            low = mid
        else:
            high = mid
    return round((low + high) / 2, 2)

In [ ]:
check([('effective_b(6, 2)', 2.0), ('effective_b(52, 5)', 1.92), ('effective_b(14, 3)', 2.0), ('effective_b(57, 12)', 1.23)])